# Adult Income - Data Preprocessing

This notebook focuses on feature engineering and preprocessing. It maps the target variable, consolidates categorical classes, splits the dataset into training and testing sets, and constructs transformation pipelines (imputation, scaling, one-hot encoding) to prepare the data for modeling.

## Contents
1. Load Data
2. Check Data Info
3. Target Column Mapping
4. Combining Categorical Variables
5. Separate Features and Target
6. Train Test Split
7. Preprocessing Pipelines
8. Combine X and y into DataFrames
9. Exporting Train and Test Datasets

In [147]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler

# 1. Load Data

In [148]:
df = pd.read_csv('../data/cleaned_adult_income.csv')

# 2. Check Data Info.

In [149]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29096 entries, 0 to 29095
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             29096 non-null  int64 
 1   workclass       29096 non-null  object
 2   education.num   29096 non-null  int64 
 3   marital.status  29096 non-null  object
 4   occupation      29096 non-null  object
 5   relationship    29096 non-null  object
 6   race            29096 non-null  object
 7   sex             29096 non-null  object
 8   capital.gain    29096 non-null  int64 
 9   capital.loss    29096 non-null  int64 
 10  hours.per.week  29096 non-null  int64 
 11  native.country  29096 non-null  object
 12  income          29096 non-null  object
dtypes: int64(5), object(8)
memory usage: 2.9+ MB


# 3. Target column `Income` mapping

In [150]:
df["income"] = df["income"].map({"<=50K": 0, ">50K": 1})

# 4. Combining Categorical Vriables

Maritial Status

In [151]:
df['marital.status'] = df['marital.status'].replace({
    'Married-civ-spouse': 'Married',
    'Married-AF-spouse': 'Married'
})

df['marital.status'].value_counts()

marital.status
Married                  13272
Never-married             9173
Divorced                  4237
Separated                 1014
Widowed                    982
Married-spouse-absent      418
Name: count, dtype: int64

Relationship

In [152]:
df['relationship'] = df['relationship'].replace({
    'Husband': 'Spouse',
    'Wife': 'Spouse'
})

df['relationship'].value_counts()

relationship
Spouse            13034
Not-in-family      7684
Own-child          4096
Unmarried          3317
Other-relative      965
Name: count, dtype: int64

Workclass

In [154]:
df['workclass'] = df['workclass'].replace({
    'Without-pay': 'No-work',
    'Never-worked': 'No-work'
})

df['workclass'] = df['workclass'].replace({
    'State-gov': 'State/Local-gov',
    'Local-gov': 'State/Local-gov'
})

df['workclass'].value_counts()

workclass
Private             19621
State/Local-gov      3312
Self-emp-not-inc     2473
Unknown              1632
Self-emp-inc         1091
Federal-gov           946
No-work                21
Name: count, dtype: int64

# 5. Separate `Features` and `Target`

In [155]:
X = df.drop(columns="income")
y = df["income"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (29096, 12)
y shape: (29096,)


# 6. Trian Test Split

In [156]:
X = df.drop('income', axis=1)
y = df['income']

In [157]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

print("\nTarget distribution:")
display(pd.DataFrame({
    "train_%": y_train.value_counts(normalize=True).sort_index() * 100,
    "test_%": y_test.value_counts(normalize=True).sort_index() * 100
}))

Train: (23276, 12) (23276,)
Test : (5820, 12) (5820,)

Target distribution:


,train_%,test_%
income,,
0,75.21911,75.223368
1,24.78089,24.776632


In [158]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f'''
Numerical Columns: {len(num_cols)}
{num_cols}

Categorical Columns: {len(cat_cols)}
{cat_cols}
''')


Numerical Columns: 5
['age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

Categorical Columns: 7
['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']



# 7. Preprocessing Pipelines for Numerical and Categorical Features

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        min_frequency=100,
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ]
)

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

Numeric columns: ['age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
Categorical columns: ['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']


In [160]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Transformed train shape:", X_train_transformed.shape)
print("Transformed test shape :", X_test_transformed.shape)

Transformed train shape: (23276, 52)
Transformed test shape : (5820, 52)


In [161]:
feature_names = preprocessor.get_feature_names_out()

print("Number of generated features:", len(feature_names))
display(pd.Series(feature_names, name="feature_name").head(30))

Number of generated features: 52


0                                      num__age
1                            num__education.num
2                             num__capital.gain
3                             num__capital.loss
4                           num__hours.per.week
5                    cat__workclass_Federal-gov
6                        cat__workclass_Private
7                   cat__workclass_Self-emp-inc
8               cat__workclass_Self-emp-not-inc
9                cat__workclass_State/Local-gov
10                       cat__workclass_Unknown
11            cat__workclass_infrequent_sklearn
12                 cat__marital.status_Divorced
13                  cat__marital.status_Married
14    cat__marital.status_Married-spouse-absent
15            cat__marital.status_Never-married
16                cat__marital.status_Separated
17                  cat__marital.status_Widowed
18                 cat__occupation_Adm-clerical
19                 cat__occupation_Craft-repair
20              cat__occupation_Exec-man

# 8. Combine X and y into two DataFrames

In [162]:
train_df = pd.DataFrame(
    X_train_transformed.toarray(),
    columns=feature_names
)
train_df["income"] = y_train.values

In [163]:
test_df = pd.DataFrame(
    X_test_transformed.toarray(),
    columns=feature_names
)
test_df["income"] = y_test.values

# 9. Exporting Train and Test Datasets

In [164]:
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)